In [1]:
import netCDF4
import zipfile
import os
import zipfile
import tarfile
import xarray as xr
import os
import shutil
from typing import Dict, Any, Optional


In [2]:
#!/usr/bin/env python3
"""
Script d'analyse complète de fichiers NetCDF
Gère l'analyse d'un seul fichier ou de tous les fichiers .nc dans un dossier.
"""

import os
import netCDF4 as nc
import numpy as np
import sys
# Note: datetime et timedelta ne sont pas utilisés directement, mais conservés pour la complétude
from datetime import datetime, timedelta 
import shutil


def analyser_fichier_unique(fichier_complet, afficher_apercu):
    """ Analyse et affiche les métadonnées pour un seul fichier NetCDF. """
    nom_fichier = os.path.basename(fichier_complet)
    
    print("=" * 100)
    print(f"📂 FICHIER: {nom_fichier}")
    print("=" * 100)
    
    try:
        ds = nc.Dataset(fichier_complet, 'r')
        
        # ==================== INFORMATIONS GÉNÉRALES ====================
        print("\n📋 INFORMATIONS GÉNÉRALES")
        print("-" * 100)
        print(f"  Format NetCDF     : {ds.data_model}")
        print(f"  Taille du fichier : {os.path.getsize(fichier_complet) / (1024**2):.2f} Mo")
        print(f"  Chemin complet    : {fichier_complet}")
        
        # ==================== DIMENSIONS ====================
        print("\n📏 DIMENSIONS")
        print("-" * 100)
        if not ds.dimensions:
            print("  Aucune dimension trouvée")
        else:
            for dim_name, dim in ds.dimensions.items():
                taille = f"UNLIMITED (actuellement {len(dim)})" if dim.isunlimited() else len(dim)
                print(f"  • {dim_name:20s}: {taille}")
        
        # ==================== VARIABLES ====================
        print(f"\n📊 VARIABLES ({len(ds.variables)})")
        print("-" * 100)
        
        for var_name, var in ds.variables.items():
            print(f"\n  ▸ Variable: {var_name}")
            print(f"    ├─ Type       : {var.dtype}")
            print(f"    ├─ Dimensions : {var.dimensions}")
            print(f"    ├─ Forme      : {var.shape}")
            print(f"    ├─ Taille     : {var.size} éléments")
            
            # Attributs de la variable
            if var.ncattrs():
                print(f"    └─ Attributs  :")
                for attr_name in var.ncattrs():
                    attr_value = getattr(var, attr_name)
                    if isinstance(attr_value, str) and len(attr_value) > 80:
                        attr_value = attr_value[:80] + "..."
                    print(f"       • {attr_name}: {attr_value}")
            
            # Statistiques sur les données (si possible)
            if afficher_apercu and var.size > 0 and var.size < 1e8 and np.issubdtype(var.dtype, np.number):
                try:
                    data = var[:]
                    # Gestion des masques et NaN
                    if hasattr(data, 'mask'):
                        data_valide = data[~data.mask]
                    else:
                        data_valide = data[~np.isnan(data)] if np.any(np.isnan(data)) else data
                    
                    if len(data_valide) > 0:
                        print(f"    📊 Statistiques:")
                        print(f"       • Min     : {np.min(data_valide):.6f}")
                        print(f"       • Max     : {np.max(data_valide):.6f}")
                        print(f"       • Moyenne : {np.mean(data_valide):.6f}")
                        # Valeurs manquantes
                        if hasattr(data, 'mask') and data.size > 0:
                            pct_manquant = (np.sum(data.mask) / data.size) * 100
                            print(f"       • Valeurs manquantes: {np.sum(data.mask)} ({pct_manquant:.2f}%)")
                    
                    # Aperçu des premières valeurs
                    if var.ndim == 1:
                        if var.size <= 20:
                            print(f"    🔢 Valeurs: {data}")
                        else:
                            print(f"    🔢 Premières valeurs: {data[:5]}")
                            print(f"       Dernières valeurs: {data[-5:]}")
                except Exception as e:
                    print(f"    ⚠️  Impossible d'accéder aux données: {e}")

        # ==================== ATTRIBUTS GLOBAUX ====================
        print(f"\n ATTRIBUTS GLOBAUX ({len(ds.ncattrs())})")
        print("-" * 100)
        if not ds.ncattrs():
            print("  Aucun attribut global trouvé")
        else:
            for attr_name in ds.ncattrs():
                attr_value = getattr(ds, attr_name)
                if isinstance(attr_value, str) and len(attr_value) > 200:
                    attr_value = attr_value[:200] + "..."
                print(f"  • {attr_name:30s}: {attr_value}")
        
        # RÉSOUDRE LAT/LON et TEMPS (Le code original pour ces sections est fonctionnel et réutilisé ici)
        # ==================== RÉSOLUTION SPATIALE ====================
        print("\n RÉSOLUTION SPATIALE")
        print("-" * 100)
        lat_keys = [k for k in ds.variables.keys() if 'lat' in k.lower()]
        lon_keys = [k for k in ds.variables.keys() if 'lon' in k.lower()]
        
        if lat_keys and lon_keys:
            try:
                lat = ds.variables[lat_keys[0]][:]
                lon = ds.variables[lon_keys[0]][:]
                
                if len(lat) > 1:
                    res_lat = np.abs(np.mean(np.diff(lat)))
                    print(f"  • Latitude  : {lat_keys[0]}")
                    print(f"    - Plage       : {np.min(lat):.6f}° à {np.max(lat):.6f}°")
                    print(f"    - Résolution  : {res_lat:.6f}° (~{res_lat * 111:.2f} km)")
                    print(f"    - Nombre      : {len(lat)} points")
                
                if len(lon) > 1:
                    res_lon = np.abs(np.mean(np.diff(lon)))
                    print(f"  • Longitude : {lon_keys[0]}")
                    print(f"    - Plage       : {np.min(lon):.6f}° à {np.max(lon):.6f}°")
                    print(f"    - Résolution  : {res_lon:.6f}° (~{res_lon * 111:.2f} km à l'équateur)")
                    print(f"    - Nombre      : {len(lon)} points")
                
                if len(lat) > 1 and len(lon) > 1:
                    surface_deg = (np.max(lat) - np.min(lat)) * (np.max(lon) - np.min(lon))
                    print(f"  • Couverture   : {surface_deg:.2f} degrés²")
                    print(f"  • Grille       : {len(lat)} × {len(lon)} = {len(lat) * len(lon)} points")
            
            except Exception as e:
                print(f"  ⚠️  Erreur lors de l'analyse spatiale: {e}")
        else:
            print("  ⚠️  Informations spatiales non disponibles (pas de variables lat/lon)")
        
        # ==================== INFORMATIONS TEMPORELLES ====================
        print("\n  INFORMATIONS TEMPORELLES")
        print("-" * 100)
        time_keys = [k for k in ds.variables.keys() if 'time' in k.lower()]
        
        if time_keys:
            try:
                time_var = ds.variables[time_keys[0]]
                time_data = time_var[:]
                n_pas = len(time_data)
                
                print(f"  • Variable temporelle : {time_keys[0]}")
                print(f"  • Nombre de pas       : {n_pas}")
                print(f"  • Type                : {time_var.dtype}")
                
                if hasattr(time_var, 'units'):
                    print(f"  • Unités              : {time_var.units}")
                    try:
                        dates = nc.num2date(time_data, units=time_var.units, 
                                            calendar=getattr(time_var, 'calendar', 'standard'))
                        
                        if len(dates) > 0:
                            print(f"  • Période             : {dates[0]} à {dates[-1]}")
                            
                            if len(dates) > 1:
                                deltas = np.diff([d.timestamp() if hasattr(d, 'timestamp') else 0 for d in dates[:min(10, len(dates))]])
                                if len(deltas) > 0 and np.std(deltas) < np.mean(deltas) * 0.1:
                                    intervalle_sec = np.mean(deltas)
                                    if intervalle_sec < 3600:
                                        print(f"  • Intervalle          : {intervalle_sec/60:.1f} minutes")
                                    elif intervalle_sec < 86400:
                                        print(f"  • Intervalle          : {intervalle_sec/3600:.1f} heures")
                                    else:
                                        print(f"  • Intervalle          : {intervalle_sec/86400:.1f} jours")
                                
                                if n_pas <= 10:
                                    print(f"  • Dates               : {[str(d) for d in dates]}")
                                else:
                                    print(f"  • Premières dates     : {[str(d) for d in dates[:5]]}")
                                    print(f"  • Dernières dates     : {[str(d) for d in dates[-5:]]}")
                        
                    except Exception as e:
                        print(f"  ⚠️  Impossible de décoder les dates: {e}")
                        print(f"  • Valeurs brutes      : {time_data[:min(10, len(time_data))]}")
                else:
                    print(f"  • Valeurs brutes      : {time_data[:min(20, len(time_data))]}")

            except Exception as e:
                print(f"  ⚠️  Erreur lors de l'analyse temporelle: {e}")
        else:
            print("  ⚠️  Aucune variable temporelle trouvée")
        
        # ==================== GROUPES (NetCDF4) ====================
        if hasattr(ds, 'groups') and len(ds.groups) > 0:
            print(f"\n📁 GROUPES ({len(ds.groups)})")
            print("-" * 100)
            for group_name, group in ds.groups.items():
                print(f"  • {group_name}")
                print(f"    - Dimensions : {len(group.dimensions)}")
                print(f"    - Variables  : {len(group.variables)}")
                print(f"    - Attributs  : {len(group.ncattrs())}")
        
        # ==================== RÉSUMÉ ====================
        print(f"\n📑 RÉSUMÉ")
        print("-" * 100)
        print(f"  ✓ Dimensions       : {len(ds.dimensions)}")
        print(f"  ✓ Variables        : {len(ds.variables)}")
        print(f"  ✓ Attributs globaux: {len(ds.ncattrs())}")
        
        taille_totale = sum(var.size * var.dtype.itemsize for var in ds.variables.values())
        print(f"  ✓ Taille des données: {taille_totale / (1024**2):.2f} Mo")
        
        ds.close()
        print(f"\n✅ Analyse terminée avec succès\n")
        
    except Exception as e:
        print(f"\n❌ ERREUR lors de l'analyse du fichier '{nom_fichier}':")
        print(f"   {type(e).__name__}: {e}\n")


def analyser_chemins_nc(chemin, afficher_apercu=True):
    """
    Gère l'entrée, qu'il s'agisse d'un fichier .nc ou d'un dossier.
    """
    if os.path.isfile(chemin) and chemin.endswith(".nc"):
        # CAS 1 : C'est un fichier .nc
        print(f"\n🔍 ANALYSE D'UN SEUL FICHIER .NC")
        analyser_fichier_unique(chemin, afficher_apercu)
        
    elif os.path.isdir(chemin):
        # CAS 2 : C'est un dossier
        fichiers_nc = [f for f in os.listdir(chemin) if f.endswith(".nc")]
        
        if not fichiers_nc:
            print(f"⚠️  Aucun fichier .nc trouvé dans {chemin}")
            return
            
        print(f"\n🔍 ANALYSE DE {len(fichiers_nc)} FICHIER(S) .NC")
        print(f"📁 Dossier: {chemin}\n")
        
        for idx, nom_fichier in enumerate(fichiers_nc, 1):
            fichier_complet = os.path.join(chemin, nom_fichier)
            print(f"📂 FICHIER {idx}/{len(fichiers_nc)}: {nom_fichier}")
            analyser_fichier_unique(fichier_complet, afficher_apercu)
            if idx < len(fichiers_nc):
                print("\n" + "═" * 100 + "\n")

    else:
        # CAS 3 : Ni un fichier, ni un dossier valide
        print(f"❌ Chemin invalide. Doit être un fichier .nc ou un dossier existant: '{chemin}'")


import os
import netCDF4 as nc
import numpy as np
from datetime import datetime
import re # Pour l'analyse de l'unité de temps

def corriger_et_sauvegarder(fichier_original, dossier_cible=None):
    """
    Copie le fichier original, applique les corrections de temps 
    sur la copie, et sauvegarde la copie.
    """
    
    nom_original = os.path.basename(fichier_original)
    
    # 1. Définir le chemin du nouveau fichier
    if dossier_cible is None:
        dossier_cible = os.path.dirname(fichier_original)
        
    nom_corrigé = nom_original.replace(".nc", "_corrigé.nc")
    fichier_corrigé = os.path.join(dossier_cible, nom_corrigé)
    
    print(f"\n⚙️ Démarrage de la correction pour: **{nom_original}**")
    print(f"  → Fichier corrigé cible: {nom_corrigé}")
    
    # 2. Copier le fichier original
    try:
        shutil.copyfile(fichier_original, fichier_corrigé)
        print(f"  ✅ Copie de l'original vers le nouveau fichier réussie.")
    except Exception as e:
        print(f"  ❌ ERREUR lors de la copie du fichier: {e}")
        return

    # 3. Appliquer les corrections sur la COPIE
    try:
        # Ouvrir la COPIE en mode écriture ('a' pour modifier)
        with nc.Dataset(fichier_corrigé, 'a') as ds:
            if 'Times' in ds.variables:
                time_var = ds.variables['Times']
                original_units = getattr(time_var, 'units', '')
                n_steps = len(time_var)

                # --- CAS 1: Fichier FRA01 (Valeurs 'Times' à zéro) ---
                if n_steps > 0 and np.all(time_var[:] == 0.0) and "FRA01" in nom_original:
                    print("  → Valeurs 'Times' à zéro détectées. Correction des données et des unités.")
                    match = re.search(r'\.(\d{4})\.', nom_original)
                    annee = int(match.group(1)) if match else 2016 
                    nouvelles_valeurs_temps = np.arange(0.5, n_steps + 0.5)
                    time_var[:] = nouvelles_valeurs_temps
                    time_var.units = f'days since {annee}-01-01 00:00:00'
                    print(f"  ✅ Valeurs 'Times' mises à jour. Nouvelles unités: {time_var.units}")
                
                # --- CAS 2: Fichiers FRA03 (Problème d'unité non-standard) ---
                elif 'day as %Y%m%d.%f' in original_units:
                    print(f"  → Unité non standard '{original_units}' détectée. Correction de l'unité.")
                    if n_steps > 0:
                        premiere_valeur = time_var[0]
                        annee = int(premiere_valeur // 10000)
                        
                        # Fixer la valeur de début (centre du jour)
                        valeur_debut = 11.5 / 24.0 # Environ 0.47916667
                        nouvelles_valeurs_temps = np.arange(n_steps, dtype=np.float64) + valeur_debut
                        
                        time_var[:] = nouvelles_valeurs_temps
                        time_var.units = f'days since {annee}-01-01 00:00:00'
                        print(f"  ✅ Valeurs 'Times' converties et unités mises à jour: {time_var.units}")
                    else:
                        print("  ❌ Aucune donnée temporelle à corriger.")
                
                else:
                    print(f"  ▶️ Aucune correction nécessaire sur les unités: {original_units}")
            else:
                print("  ⚠️ La variable 'Times' est manquante.")
        
        print(f"✅ Correction et sauvegarde terminées pour **{nom_corrigé}**.")

    except Exception as e:
        print(f"❌ ERREUR fatale lors de la correction de '{nom_corrigé}': {e}")


def executer_correction(chemin_a_analyser):
    """ Exécute la correction et la sauvegarde pour tous les fichiers .nc dans le dossier. """
    if not os.path.isdir(chemin_a_analyser):
        print(f"❌ Le chemin de correction est invalide ou n'existe pas: {chemin_a_analyser}")
        return
        
    fichiers_nc = sorted([f for f in os.listdir(chemin_a_analyser) if f.endswith(".nc") and not f.endswith("_corrigé.nc")])
    
    if not fichiers_nc:
        print(f"⚠️ Aucun fichier .nc original trouvé dans {chemin_a_analyser}")
        return

    print("=" * 100)
    print(f"🚀 DÉMARRAGE DU PROCESSUS DE CORRECTION (Sauvegarde séparée) ({len(fichiers_nc)} fichier(s))")
    print("=" * 100)
    
    for nom_fichier in fichiers_nc:
        fichier_complet = os.path.join(chemin_a_analyser, nom_fichier)
        # Appel de la nouvelle fonction
        corriger_et_sauvegarder(fichier_complet, chemin_a_analyser)

# ==============================================================================
# --- À EXÉCUTER DANS VOTRE ENVIRONNEMENT PYTHON APRÈS SAUVEGARDE DU FICHIER ---
# ==============================================================================

if __name__ == "__main__":
    # Utilisez le même chemin que dans votre script d'analyse
    chemin_a_analyser = r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data_temp"
    
    # Exécuter la correction
    executer_correction(chemin_a_analyser)
    
    # Relancez ensuite votre script d'analyse original pour vérifier les changements!
    print("\n\n*** CORRECTION TERMINÉE. VEUILLEZ RELANCER VOTRE SCRIPT D'ANALYSE ORIGINAL POUR VÉRIFICATION. ***")

        # Lancer l'analyse complète
    analyser_chemins_nc(chemin_a_analyser, afficher_apercu=True)


# if __name__ == "__main__":
#     # if len(sys.argv) > 1:
#     #     # Utiliser l'argument passé en ligne de commande
#     #     chemin_a_analyser = sys.argv[1]
#     # else:
#         # Utiliser le chemin par défaut (À vérifier sur votre machine!)
#     chemin_a_analyser = r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data_temp"
    


🚀 DÉMARRAGE DU PROCESSUS DE CORRECTION (Sauvegarde séparée) (5 fichier(s))

⚙️ Démarrage de la correction pour: **INERIS.REANALYSED.FRA01.2016.NO2.daymean.2gis.nc**
  → Fichier corrigé cible: INERIS.REANALYSED.FRA01.2016.NO2.daymean.2gis_corrigé.nc


KeyboardInterrupt: 

In [3]:
import os
import xarray as xr
import numpy as np

# Répertoire contenant les fichiers NetCDF
repertoire = r"R:\Direction_Data\0_Projets\Projet_CANCAIR\pneumodetect\data_air"

# Nombre de valeurs à afficher en début et fin
N = 5

In [8]:
from collections import defaultdict

# Polluants d'intérêt
polluants = ["PM10", "NO2", "O3", "PM25"]

# Dictionnaire pour stocker les fichiers par polluant
fichiers_par_polluant = defaultdict(list)

# Parcours des fichiers pour trier par polluant
for fichier in os.listdir(repertoire):
    if fichier.endswith(".nc"):
        for polluant in polluants:
            if f".{polluant}." in fichier:
                fichiers_par_polluant[polluant].append(fichier)

# Limiter à 3 fichiers par polluant
for polluant, fichiers in fichiers_par_polluant.items():
    fichiers_par_polluant[polluant] = sorted(fichiers)[:3]

# Traitement des fichiers sélectionnés
for polluant, fichiers in fichiers_par_polluant.items():
    print(f"\n==================== Traitement {polluant} ====================")
    for fichier in fichiers:
        chemin_fichier = os.path.join(repertoire, fichier)
        print("\n" + "="*60)
        print(f"📂 Fichier : {fichier}")

        ds = xr.open_dataset(chemin_fichier)

        # =================== RÉSOLUTION SPATIALE ======================
        if 'lat' in ds.coords and 'lon' in ds.coords:
            lat = ds['lat'].values
            lon = ds['lon'].values
            if len(lat) > 1 and len(lon) > 1:
                res_lat = np.abs(lat[1] - lat[0])
                res_lon = np.abs(lon[1] - lon[0])
                print(f"🗺️ Résolution spatiale :")
                print(f"   - Latitude : {res_lat:.6f} degrés")
                print(f"   - Longitude : {res_lon:.6f} degrés")
            else:
                print("⚠️ Coordonnées lat/lon insuffisantes pour calculer la résolution")
        else:
            print("⚠️ Coordonnées lat/lon non trouvées")

        # =================== ESTIMATION EN MÈTRES =====================
        if 'lat' in ds.coords and 'lon' in ds.coords and len(lat) > 1 and len(lon) > 1:
            res_lat_deg = np.abs(lat[1] - lat[0])
            res_lon_deg = np.abs(lon[1] - lon[0])
            lat_moy = np.mean(lat)
            metres_par_deg_lat = 111_320
            metres_par_deg_lon = 111_320 * np.cos(np.deg2rad(lat_moy))
            res_lat_m = res_lat_deg * metres_par_deg_lat
            res_lon_m = res_lon_deg * metres_par_deg_lon
            print("📐 Résolution estimée en mètres :")
            print(f"   - Latitude : {res_lat_m:.1f} m")
            print(f"   - Longitude : {res_lon_m:.1f} m")

        # =================== VARIABLES ===============================
        for var in ds.data_vars:
            data = ds[var]
            print("\n🔹 Variable :", var)
            print("Dimensions :", data.dims)
            print("Shape :", data.shape)

            if np.issubdtype(data.dtype, np.number):
                fill_value = data.attrs.get('_FillValue', None)
                values = data.values.flatten()

                if fill_value is not None:
                    nb_null = np.isnan(values).sum() + np.sum(values == fill_value)
                    values_nonnull = values[(~np.isnan(values)) & (values != fill_value)]
                else:
                    nb_null = np.isnan(values).sum()
                    values_nonnull = values[~np.isnan(values)]

                print("Nombre de valeurs nulles :", nb_null)

                if values_nonnull.size > 0:
                    print(f"Premières {N} valeurs :", values_nonnull[:N])
                    print(f"Dernières {N} valeurs :", values_nonnull[-N:])
                else:
                    print("⚠️ Variable vide ou uniquement des valeurs nulles")

            else:
                flattened = data.values.flatten()
                print("⚠️ Variable non numérique")
                print(f"Premières {N} valeurs :", flattened[:N])
                print(f"Dernières {N} valeurs :", flattened[-N:])

        ds.close()



==================== Traitement NO2 ====================

📂 Fichier : INERIS.REANALYSED.FRA01.2016.NO2.daymean.2gis.nc
🗺️ Résolution spatiale :
   - Latitude : 0.007810 degrés
   - Longitude : 0.015625 degrés
📐 Résolution estimée en mètres :
   - Latitude : 869.4 m
   - Longitude : 1199.2 m

🔹 Variable : NO2
Dimensions : ('Times', 'lat', 'lon')
Shape : (366, 1521, 1081)
Nombre de valeurs nulles : 406589766
Premières 5 valeurs : [10.84364 10.87333 10.55913 10.58097 10.23329]
Dernières 5 valeurs : [25.78341 25.6488  26.54578 26.22262 25.79137]

📂 Fichier : INERIS.REANALYSED.FRA01.2017.NO2.daymean.2gis.nc
🗺️ Résolution spatiale :
   - Latitude : 0.007810 degrés
   - Longitude : 0.015625 degrés
📐 Résolution estimée en mètres :
   - Latitude : 869.4 m
   - Longitude : 1199.2 m

🔹 Variable : NO2
Dimensions : ('Times', 'lat', 'lon')
Shape : (365, 1521, 1081)
Nombre de valeurs nulles : 405478865
Premières 5 valeurs : [19.26719 19.32497 19.13062 19.18904 18.98584]
Dernières 5 valeurs : [5.0654

In [17]:
import xarray as xr
import numpy as np
from pathlib import Path

filepath = r'R:\Direction_Data\0_Projets\Projet_CANCAIR\pneumodetect\data_air\INERIS.REANALYSED.FRA03.2009.PM10.daymean.2gis.nc'

def get_resolution_report(path):
    ds = xr.open_dataset(path)
    filename = Path(path).name
    
    # 1. Identification de la variable principale (souvent la seule qui n'est pas une coordonnée)
    data_vars = [v for v in ds.data_vars if v not in ['lat', 'lon', 'time']]
    var_name = data_vars[0] if data_vars else "Inconnue"
    var_data = ds[var_name]

    # 2. Calcul de la résolution en degrés
    # On calcule la différence entre deux points consécutifs
    res_lat = abs(float(ds.lat[1] - ds.lat[0]))
    res_lon = abs(float(ds.lon[1] - ds.lon[0]))

    # 3. Estimation en mètres
    # Formule simplifiée : 1° lat ≈ 111.1 km | 1° lon ≈ 111.1 km * cos(lat)
    lat_moyenne = float(ds.lat.mean())
    res_lat_m = res_lat * 111132.95  # approximation à l'équateur
    res_lon_m = res_lon * 111132.95 * np.cos(np.radians(lat_moyenne))

    print(f"📂 Fichier : {filename}")
    print(f"🗺️ Résolution spatiale :")
    print(f"   - Latitude  : {res_lat:.6f} degrés")
    print(f"   - Longitude : {res_lon:.6f} degrés")
    print(f"📐 Résolution estimée en mètres :")
    print(f"   - Latitude  : {res_lat_m:.1f} m")
    print(f"   - Longitude : {res_lon_m:.1f} m")
    print(f"\n🔹 Variable : {var_name}")
    print(f"Dimensions : {var_data.dims}")
    print(f"Shape : {var_data.shape}")
    
    # Statistiques rapides sur les valeurs
    # On aplatit pour les 5 premières/dernières valeurs valides (non-NaN)
    values = var_data.values.flatten()
    valid_values = values[~np.isnan(values)]
    
    print(f"Nombre de valeurs nulles (NaN) : {np.isnan(values).sum()}")
    if len(valid_values) > 0:
        print(f"Premières 5 valeurs : {valid_values[:5]}")
        print(f"Dernières 5 valeurs : {valid_values[-5:]}")
    
    ds.close()

get_resolution_report(filepath)

📂 Fichier : INERIS.REANALYSED.FRA03.2009.PM10.daymean.2gis.nc
🗺️ Résolution spatiale :
   - Latitude  : 0.031260 degrés
   - Longitude : 0.062500 degrés
📐 Résolution estimée en mètres :
   - Latitude  : 3474.0 m
   - Longitude : 4803.1 m

🔹 Variable : PM10
Dimensions : ('Times', 'lat', 'lon')
Shape : (365, 324, 240)
Nombre de valeurs nulles (NaN) : 15941375
Premières 5 valeurs : [27.382652 27.165646 27.708166 27.47066  27.29847 ]
Dernières 5 valeurs : [26.541798 26.374516 27.25092  26.5435   26.02851 ]


In [13]:
import netCDF4 as nc
import numpy as np

filepath = r'R:\Direction_Data\0_Projets\Projet_CANCAIR\pneumodetect\data_air\INERIS.REANALYSED.FRA01.2016.NO2.daymean.2gis.nc'

ds = nc.Dataset(filepath)
no2_data = ds.variables['NO2'][:]  # Shape: (365, 1521, 1081)

# Vérifier que les valeurs changent entre les jours
print("Jour 0 (1er janvier) - Pixel (500, 500):", no2_data[0, 500, 500])
print("Jour 1 (2 janvier) - Pixel (500, 500):", no2_data[1, 500, 500])
print("Jour 2 (3 janvier) - Pixel (500, 500):", no2_data[2, 500, 500])

# Vérifier la variance temporelle
pixel_timeseries = no2_data[:, 500, 500]
print(f"\nVariance temporelle pour pixel (500,500): {np.var(pixel_timeseries):.2f}")
print(f"Min: {np.min(pixel_timeseries):.2f}, Max: {np.max(pixel_timeseries):.2f}")

# Si variance = 0 → Les données sont identiques (problème source)
# Si variance > 0 → Le problème est dans l'extraction (code)

ds.close()

Jour 0 (1er janvier) - Pixel (500, 500): 10.92798
Jour 1 (2 janvier) - Pixel (500, 500): 10.94467
Jour 2 (3 janvier) - Pixel (500, 500): 13.60846

Variance temporelle pour pixel (500,500): 24.80
Min: 0.45, Max: 29.25


In [ ]:
"""
EXPORT TABLE POSTGRESQL → CSV (1 SEUL FICHIER)
===============================================
Exporte TOUTE la table pneumodetect_temp_air_ineris en 1 CSV
"""

import pandas as pd
from sqlalchemy import create_engine, text
import time
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Connexion PostgreSQL
DB_CREDS = {
    "database": "sandbox",
    "host": "192.168.64.142",
    "password": "vuNMRtfQf08ZTko9h4ts",
    "port": "5432",
    "user": "cancair"
}

# Nom de la table à exporter
TABLE_NAME = 'pneumodetect_temp_air_ineris'

# Fichier de sortie
OUTPUT_FILE = r'C:\Users\lpokambo\Desktop\pneumodetect_temp_air_ineris.csv'

print("="*80)
print("EXPORT POSTGRESQL → CSV (COMPLET)")
print("="*80)

# ============================================================================
# CONNEXION
# ============================================================================

print(f"\n🔌 Connexion à PostgreSQL...")

engine = create_engine(
    f"postgresql://{DB_CREDS['user']}:{DB_CREDS['password']}"
    f"@{DB_CREDS['host']}:{DB_CREDS['port']}/{DB_CREDS['database']}"
)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("   ✅ Connecté !\n")
except Exception as e:
    print(f"   ❌ Erreur connexion: {e}")
    exit(1)

# ============================================================================
# ANALYSE DE LA TABLE
# ============================================================================

print(f"📊 Analyse de la table {TABLE_NAME}...")

try:
    with engine.connect() as conn:
        # Statistiques
        result = conn.execute(text(f"""
            SELECT 
                COUNT(*) as nb_lignes,
                COUNT(DISTINCT pseudo_provisoire) as nb_patients,
                MIN(date) as date_min,
                MAX(date) as date_max
            FROM {TABLE_NAME}
        """)).fetchone()
        
        nb_lignes = result[0]
        nb_patients = result[1]
        date_min = result[2]
        date_max = result[3]
        
        print(f"\n   ✅ Table trouvée")
        print(f"   Total lignes: {nb_lignes:,}")
        print(f"   Patients: {nb_patients:,}")
        print(f"   Période: {date_min} → {date_max}")
        
        # Colonnes
        colonnes = conn.execute(text(f"""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_name = '{TABLE_NAME}'
            ORDER BY ordinal_position
        """)).fetchall()
        
        cols_list = [c[0] for c in colonnes]
        print(f"   Colonnes ({len(cols_list)}): {', '.join(cols_list)}")
        
        # Estimation taille fichier
        avg_row_size = 100  # bytes estimé par ligne
        estimated_size_mb = (nb_lignes * avg_row_size) / (1024 * 1024)
        
        print(f"\n⚠️  Taille estimée du fichier CSV: {estimated_size_mb:.1f} MB")
        
        if estimated_size_mb > 500:
            print(f"   ⚠️  ATTENTION: Fichier volumineux (> 500 MB) !")
        
except Exception as e:
    print(f"   ❌ Erreur: {e}")
    exit(1)

# ============================================================================
# CONFIRMATION
# ============================================================================

print(f"\n📁 Fichier de sortie: {OUTPUT_FILE}")
print(f"   Format: CSV avec séparateur point-virgule (;)")

confirmation = input(f"\n👉 Confirmer l'export de {nb_lignes:,} lignes ? (o/n) : ")

if confirmation.lower() != 'o':
    print("\n❌ Export annulé")
    exit(0)

# ============================================================================
# EXPORT PAR CHUNKS (pour ne pas saturer la mémoire)
# ============================================================================

print(f"\n📥 Export en cours...")
print(f"   ⏱️  Cela peut prendre plusieurs minutes...")
print(f"   💡 Export par chunks de 500,000 lignes\n")

start = time.time()

CHUNK_SIZE = 500000
nb_chunks = (nb_lignes + CHUNK_SIZE - 1) // CHUNK_SIZE

first_chunk = True

for chunk_num in range(nb_chunks):
    offset = chunk_num * CHUNK_SIZE
    
    # Barre de progression
    progress = ((chunk_num + 1) / nb_chunks) * 100
    print(f"   📊 Chunk {chunk_num + 1}/{nb_chunks} ({progress:.1f}%) - Lignes {offset:,} à {min(offset+CHUNK_SIZE, nb_lignes):,}...", end='\r')
    
    # Charger le chunk
    df_chunk = pd.read_sql(
        f"""SELECT * FROM {TABLE_NAME} 
            ORDER BY date, pseudo_provisoire 
            LIMIT {CHUNK_SIZE} OFFSET {offset}""",
        engine
    )
    
    # Écrire dans le fichier
    df_chunk.to_csv(
        OUTPUT_FILE,
        index=False,
        sep=';',
        mode='w' if first_chunk else 'a',
        header=first_chunk,
        encoding='utf-8-sig'  # Pour Excel
    )
    
    first_chunk = False

duration = time.time() - start

print(f"\n\n   ✅ Export terminé en {duration:.1f} secondes ({duration/60:.1f} minutes)")

# ============================================================================
# VÉRIFICATION FICHIER
# ============================================================================

print(f"\n📄 Vérification du fichier...")

if os.path.exists(OUTPUT_FILE):
    file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    
    print(f"   ✅ Fichier créé avec succès")
    print(f"   Taille: {file_size_mb:.1f} MB")
    print(f"   Emplacement: {OUTPUT_FILE}")
    
    # Lire les premières lignes pour vérifier
    print(f"\n🔍 Aperçu des 3 premières lignes:")
    df_sample = pd.read_csv(OUTPUT_FILE, sep=';', nrows=3)
    print(df_sample)
    
else:
    print(f"   ❌ Erreur: Fichier non créé")

# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n" + "="*80)
print("✅ EXPORT TERMINÉ !")
print("="*80)
print(f"\n⏱️  Durée totale: {duration/60:.1f} minutes")
print(f"📊 Statistiques:")
print(f"   Lignes exportées: {nb_lignes:,}")
print(f"   Patients: {nb_patients:,}")
print(f"   Période: {date_min} → {date_max}")
print(f"   Colonnes: {len(cols_list)}")
print(f"\n📁 Fichier:")
print(f"   Nom: {os.path.basename(OUTPUT_FILE)}")
print(f"   Taille: {file_size_mb:.1f} MB")
print(f"   Emplacement: {OUTPUT_FILE}")
print(f"\n💡 Pour ouvrir:")
print(f"   - Excel: Ouvrir → Choisir 'Fichiers texte' → Séparateur ';'")
print(f"   - Python: pd.read_csv('{OUTPUT_FILE}', sep=';')")
print(f"   - R: read.csv2('{OUTPUT_FILE}')")
print("="*80)

TEST CONNEXION POSTGRESQL

🔌 TEST 1 : Création de la connexion...
   ✅ Engine SQLAlchemy créé

🔌 TEST 2 : Connexion basique...


In [ ]:
"""
EXPORT AVEC DIAGNOSTIC CONNEXION
=================================
"""

import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
import time
import os
import sys

# ============================================================================
# CONFIGURATION
# ============================================================================

DB_PARAMS = {
    "host": "192.168.64.142",
    "port": "5432",
    "database": "sandbox",
    "user": "cancair",
    "password": "vuNMRtfQf08ZTko9h4ts",
    "connect_timeout": 10  # Timeout de 10 secondes
}

TABLE_NAME = 'pneumodetect_temp_air_ineris'
OUTPUT_FILE = r'C:\Users\lpokambo\Desktop\pneumodetect_temp_air_ineris.csv'

print("="*80)
print("EXPORT POSTGRESQL → CSV (AVEC DIAGNOSTIC)")
print("="*80)

# ============================================================================
# TEST CONNEXION RAPIDE AVEC PSYCOPG2
# ============================================================================

print(f"\n🔌 Test connexion PostgreSQL...")
print(f"   Host: {DB_PARAMS['host']}")
print(f"   Port: {DB_PARAMS['port']}")
print(f"   Database: {DB_PARAMS['database']}")
print(f"   Timeout: {DB_PARAMS['connect_timeout']}s\n")

start = time.time()

try:
    print("   Tentative 1/3...", end='')
    conn = psycopg2.connect(**DB_PARAMS)
    duration = time.time() - start
    print(f" ✅ Connecté en {duration:.1f}s")
    
    # Test requête
    cursor = conn.cursor()
    cursor.execute("SELECT 1")
    cursor.close()
    conn.close()
    
    print("   ✅ Connexion fonctionnelle !\n")
    
except psycopg2.OperationalError as e:
    duration = time.time() - start
    print(f" ❌ Échec après {duration:.1f}s")
    print(f"\n❌ ERREUR: {e}")
    
    if "timeout" in str(e).lower():
        print("\n💡 DIAGNOSTIC:")
        print("   Le serveur ne répond pas dans les 10 secondes")
        print("\n🔧 SOLUTIONS:")
        print("   1. Vérifier que vous êtes sur le VPN Curie")
        print("   2. Vérifier que le serveur PostgreSQL est démarré")
        print("   3. Tester avec PowerShell:")
        print(f"      Test-NetConnection -ComputerName {DB_PARAMS['host']} -Port {DB_PARAMS['port']}")
    
    sys.exit(1)

except Exception as e:
    print(f" ❌ Erreur")
    print(f"\n❌ ERREUR: {e}")
    sys.exit(1)

# ============================================================================
# CONNEXION SQLALCHEMY
# ============================================================================

print("📊 Création engine SQLAlchemy...")

try:
    engine = create_engine(
        f"postgresql://{DB_PARAMS['user']}:{DB_PARAMS['password']}"
        f"@{DB_PARAMS['host']}:{DB_PARAMS['port']}/{DB_PARAMS['database']}",
        connect_args={'connect_timeout': 10}
    )
    
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    
    print("   ✅ Engine créé\n")

except Exception as e:
    print(f"   ❌ Erreur: {e}")
    sys.exit(1)

# ============================================================================
# ANALYSE TABLE
# ============================================================================

print(f"📊 Analyse table {TABLE_NAME}...")

try:
    with engine.connect() as conn:
        result = conn.execute(text(f"""
            SELECT 
                COUNT(*) as nb_lignes,
                COUNT(DISTINCT pseudo_provisoire) as nb_patients,
                MIN(date) as date_min,
                MAX(date) as date_max
            FROM {TABLE_NAME}
        """)).fetchone()
        
        nb_lignes = result[0]
        nb_patients = result[1]
        date_min = result[2]
        date_max = result[3]
        
        print(f"   Total lignes: {nb_lignes:,}")
        print(f"   Patients: {nb_patients:,}")
        print(f"   Période: {date_min} → {date_max}")
        
        estimated_size_mb = (nb_lignes * 100) / (1024 * 1024)
        estimated_time_min = nb_lignes / 100000  # ~100k lignes/min
        
        print(f"\n   Taille estimée: {estimated_size_mb:.1f} MB")
        print(f"   Durée estimée: {estimated_time_min:.1f} minutes")

except Exception as e:
    print(f"   ❌ Erreur: {e}")
    sys.exit(1)

# ============================================================================
# CONFIRMATION
# ============================================================================

print(f"\n📁 Fichier: {OUTPUT_FILE}")
confirmation = input(f"\n👉 Exporter {nb_lignes:,} lignes ? (o/n) : ")

if confirmation.lower() != 'o':
    print("\n❌ Export annulé")
    sys.exit(0)

# ============================================================================
# EXPORT
# ============================================================================

print(f"\n📥 Export en cours...\n")

start_export = time.time()
CHUNK_SIZE = 500000
nb_chunks = (nb_lignes + CHUNK_SIZE - 1) // CHUNK_SIZE
first_chunk = True

for chunk_num in range(nb_chunks):
    offset = chunk_num * CHUNK_SIZE
    chunk_start = time.time()
    
    try:
        # Charger chunk
        df_chunk = pd.read_sql(
            f"""SELECT * FROM {TABLE_NAME} 
                ORDER BY date, pseudo_provisoire 
                LIMIT {CHUNK_SIZE} OFFSET {offset}""",
            engine
        )
        
        # Écrire
        df_chunk.to_csv(
            OUTPUT_FILE,
            index=False,
            sep=';',
            mode='w' if first_chunk else 'a',
            header=first_chunk,
            encoding='utf-8-sig'
        )
        
        chunk_duration = time.time() - chunk_start
        progress = ((chunk_num + 1) / nb_chunks) * 100
        
        print(f"   ✅ Chunk {chunk_num + 1}/{nb_chunks} ({progress:.1f}%) - {len(df_chunk):,} lignes en {chunk_duration:.1f}s")
        
        first_chunk = False
        
    except Exception as e:
        print(f"\n   ❌ Erreur chunk {chunk_num + 1}: {e}")
        sys.exit(1)

duration_total = time.time() - start_export

# ============================================================================
# VÉRIFICATION
# ============================================================================

print(f"\n📄 Vérification...")

if os.path.exists(OUTPUT_FILE):
    file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    print(f"   ✅ Fichier créé: {file_size_mb:.1f} MB")
    
    # Aperçu
    df_sample = pd.read_csv(OUTPUT_FILE, sep=';', nrows=3)
    print(f"\n🔍 Aperçu:")
    print(df_sample)

# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n" + "="*80)
print("✅ EXPORT TERMINÉ !")
print("="*80)
print(f"\n⏱️  Durée: {duration_total/60:.1f} minutes")
print(f"📊 Lignes: {nb_lignes:,}")
print(f"📁 Fichier: {OUTPUT_FILE} ({file_size_mb:.1f} MB)")
print("="*80)

EXPORT POSTGRESQL → CSV (AVEC DIAGNOSTIC)

🔌 Test connexion PostgreSQL...
   Host: 192.168.64.142
   Port: 5432
   Database: sandbox
   Timeout: 10s

   Tentative 1/3... ❌ Échec après 10.0s

❌ ERREUR: connection to server at "192.168.64.142", port 5432 failed: timeout expired


💡 DIAGNOSTIC:
   Le serveur ne répond pas dans les 10 secondes

🔧 SOLUTIONS:
   1. Vérifier que vous êtes sur le VPN Curie
   2. Vérifier que le serveur PostgreSQL est démarré
   3. Tester avec PowerShell:
      Test-NetConnection -ComputerName 192.168.64.142 -Port 5432


SystemExit: 1

In [ ]:
import os

# Fichiers à supprimer
fichiers_a_supprimer = [
    r"R:\Direction_Data\0_Projets\Projet_CANCAIR\Canc_air_stage_ete_Loice\Airparif\output\checkpoint_temp_state.pkl",
    r"R:\Direction_Data\0_Projets\Projet_CANCAIR\Canc_air_stage_ete_Loice\Airparif\output\temp_avant_enrichissement.parquet",
    r"R:\Direction_Data\0_Projets\Projet_CANCAIR\Canc_air_stage_ete_Loice\Airparif\output\temp_apres_enrichissement.parquet",
    r"R:\Direction_Data\0_Projets\Projet_CANCAIR\Canc_air_stage_ete_Loice\Airparif\output\patients_temp_daily.parquet",
    r"R:\Direction_Data\0_Projets\Projet_CANCAIR\Canc_air_stage_ete_Loice\Airparif\output\patients_temp_mensuel.parquet",
]

for f in fichiers_a_supprimer:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ Supprimé: {os.path.basename(f)}")
    else:
        print(f"  Absent: {os.path.basename(f)}")

print("\n🔄 Relancez maintenant temp_dataset_extraction.py")

✓ Supprimé: checkpoint_temp_state.pkl
✓ Supprimé: temp_avant_enrichissement.parquet
✓ Supprimé: temp_apres_enrichissement.parquet
✓ Supprimé: patients_temp_daily.parquet
✓ Supprimé: patients_temp_mensuel.parquet

🔄 Relancez maintenant temp_dataset_extraction.py
